In [1]:
import sys, importlib.util, dataclasses
from pathlib import Path
from datetime import date
import pandas as pd

for _c in [Path.cwd()] + list(Path.cwd().parents):
    if (_c / "gbp").is_dir() and (_c / "tests").is_dir():
        sys.path.insert(0, str(_c)); break

_spec = importlib.util.spec_from_file_location("fixtures", _c / "tests/unit/build/fixtures.py")
_mod = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(_mod)
minimal_raw_model = _mod.minimal_raw_model

from gbp.core.enums import FacilityRole, FacilityType, ModalType, OperationType, PeriodType, AttributeKind, ResourceStatus
from gbp.core.roles import DEFAULT_ROLES, derive_roles
from gbp.core.model import RawModelData, ResolvedModelData
from gbp.build.pipeline import _ensure_edges_and_commodities, build_model
from gbp.build.validation import validate_raw_model
from gbp.build.time_resolution import resolve_all_time_varying, resolve_registry_attributes
from gbp.build.lead_time import resolve_lead_times
from gbp.build.transformation import resolve_transformations
from gbp.build.fleet_capacity import compute_fleet_capacity
from gbp.build.spine import assemble_spines
from gbp.consumers.simulator.state import init_state
from gbp.loaders import DataLoaderMock, DataLoaderGraph, GraphLoaderConfig
from gbp.loaders.contracts import (
    DepotsSourceSchema,
    GraphLoaderConfig,
    ResourcesSourceSchema,
    StationsSourceSchema,
)
from gbp.core.attributes.registry import AttributeRegistry

In [4]:
import numpy as np

mock_config = {
    "n_stations": 10,      # 10 stations
    "n_depots": 2,         # 2 depots
    "n_timestamps": 168,   # 1 week of hourly data
    "time_freq": "h",
    "start_date": "2025-01-01",
    "num_resources": 3,
    "resource_capacity": 100,
    "ebike_fraction": 0.3,
    "depot_capacity": 200,
    "seed": 42,
}

# Loading of mock data (step by step, matching DataLoaderMock.load_data)
mock = DataLoaderMock(mock_config)

n_stations = mock.config["n_stations"]
n_depots = mock.config.get("n_depots", 2)
n_timestamps = mock.config.get("n_timestamps", 168)
start_date = mock.config.get("start_date", "2025-01-01")
freq = mock.config.get("time_freq", "h")
ebike_fraction = float(mock.config.get("ebike_fraction", 0.3))
depot_capacity = mock.config.get("depot_capacity", 200)

# Stations — slim public table + internal full version for telemetry
full_stations = mock._generate_stations_full(n_stations)
mock.df_stations = full_stations[["station_id", "lat", "lon"]].copy()

# Station capacities per commodity_category
total_caps = full_stations["inventory_capacity"].values
ebike_caps = np.maximum(1, np.round(total_caps * ebike_fraction).astype(int))
classic_caps = total_caps - ebike_caps
cap_rows = []
for i, sid in enumerate(full_stations["station_id"]):
    cap_rows.append({"station_id": sid, "commodity_category": "electric_bike", "capacity": int(ebike_caps[i])})
    cap_rows.append({"station_id": sid, "commodity_category": "classic_bike", "capacity": int(classic_caps[i])})
mock.df_station_capacities = pd.DataFrame(cap_rows)

# Depots + depot capacities per commodity_category
mock.df_depots = mock._generate_depots(n_depots)
depot_ebike_cap = max(1, round(depot_capacity * ebike_fraction))
depot_classic_cap = depot_capacity - depot_ebike_cap
depot_cap_rows = []
for nid in mock.df_depots["node_id"]:
    depot_cap_rows.append({"node_id": nid, "commodity_category": "electric_bike", "capacity": depot_ebike_cap})
    depot_cap_rows.append({"node_id": nid, "commodity_category": "classic_bike", "capacity": depot_classic_cap})
mock.df_depot_capacities = pd.DataFrame(depot_cap_rows)

# Resources — capacity is a separate table
num_resources = mock.config.get("num_resources", 3)
resource_cap = mock.config.get("resource_capacity", 100)
resource_ids = [f"truck_{i + 1}" for i in range(num_resources)]
mock.df_resources = pd.DataFrame({"resource_id": resource_ids})
mock.df_resource_capacities = pd.DataFrame({"resource_id": resource_ids, "capacity": [resource_cap] * num_resources})

# Timestamps + inventory/telemetry/trips (with multi-commodity + depot inventory)
mock.timestamps = pd.date_range(start=start_date, periods=n_timestamps, freq=freq)
mock.df_inventory_ts, mock.df_telemetry_ts, mock.df_trips = mock._generate_inventory_telemetry_trips(
    df_stations=full_stations,
    df_depots=mock.df_depots,
    timestamps=mock.timestamps,
    ebike_fraction=ebike_fraction,
    depot_ebike_cap=depot_ebike_cap,
    depot_classic_cap=depot_classic_cap,
)

# Costs
mock.df_station_costs, mock.df_depot_costs, mock.df_truck_rates = mock._generate_costs(
    df_stations=mock.df_stations,
    df_depots=mock.df_depots,
    df_resource_capacities=mock.df_resource_capacities,
)

# Loading of RawModelData using DataLoaderGraph based on the mock data
loader = DataLoaderGraph(mock, GraphLoaderConfig())

StationsSourceSchema.validate(loader._source.df_stations)
DepotsSourceSchema.validate(loader._source.df_depots)
ResourcesSourceSchema.validate(loader._source.df_resources)

temporal = loader._build_temporal()
entities = loader._build_entities()
behavior = loader._build_behavior(entities)
distance_data = loader._build_distance_matrix(entities) if loader._config.build_edges else {}
resources = loader._build_resources(entities)
registry = AttributeRegistry()
flow_data = loader._build_node_parameters(entities, registry)
loader._register_facility_costs(registry, temporal)
loader._register_resource_costs(registry, entities)

# Observations: trips → observed_flow, telemetry → observed_inventory
observations = loader._build_observations(entities)
observed_flow = observations.get("observed_flow")
if observed_flow is not None and not observed_flow.empty:
    demand_from_obs = loader._build_demand_from_observations(observed_flow)
    if demand_from_obs is not None:
        observations["demand"] = demand_from_obs
    supply_from_obs = loader._build_supply_from_observations(observed_flow)
    if supply_from_obs is not None:
        observations["supply"] = supply_from_obs

all_tables = {
    **temporal,
    **entities.tables,
    **behavior,
    **distance_data,
    **flow_data,
    **resources,
    **{k: v for k, v in observations.items() if v is not None},
}
loader._raw = RawModelData(
        **{k: v for k, v in all_tables.items() if v is not None},
        attributes=registry,
    )


# Validation of RawModelData 
loader._raw.validate()
validate_raw_model(loader._raw).raise_if_invalid()
periods = loader._raw.periods.copy()
resolved_time = resolve_all_time_varying(loader._raw, periods)
resolved_attrs = resolve_registry_attributes(loader._raw.attributes, periods)
edges_df, ec_df = _ensure_edges_and_commodities(loader._raw)
edge_lead_time_resolved: pd.DataFrame | None = None
if edges_df is not None and not edges_df.empty:
    elt = resolve_lead_times(edges_df, periods)
    edge_lead_time_resolved = elt if not elt.empty else None
transformation_resolved = resolve_transformations(
    loader._raw.facilities,
    loader._raw.transformations,
    loader._raw.transformation_inputs,
    loader._raw.transformation_outputs,
)
fleet_capacity = compute_fleet_capacity(
    loader._raw.resource_fleet,
    loader._raw.resource_categories,
    loader._raw.resources,
)
resolved = ResolvedModelData.from_raw(
    loader._raw,
    periods=periods,
    resolved_time=resolved_time,
    resolved_attrs=resolved_attrs,
    edges=edges_df,
    edge_commodities=ec_df,
    edge_lead_time_resolved=edge_lead_time_resolved,
    transformation_resolved=transformation_resolved,
    fleet_capacity=fleet_capacity,
)

spines = assemble_spines(resolved)
resolved.facility_spines = spines["facility"] or None
resolved.edge_spines = spines["edge"] or None
resolved.resource_spines = spines["resource"] or None

2026-04-07 19:44:03 [debug    ] observed_flow_built            loader=graph_core rows=723
2026-04-07 19:44:03 [debug    ] observed_inventory_built       loader=graph_core rows=140


In [126]:
df = mock.df_inventory_ts[('station_1', 'electric_bike')].reset_index()
df['index'] = df['index'].dt.date
df.groupby('index').last()

df = mock.df_trips[['started_at', 'start_station_id', 'rideable_type']]
df['started_at'] = df['started_at'].dt.date
df.groupby(['started_at', 'start_station_id', 'rideable_type']).size().reset_index()[:10]

df = mock.df_telemetry_ts[['timestamp', 'station_id', 'num_bikes_available', 'num_ebikes_available']]
df['timestamp'] = df['timestamp'].dt.date
# df = df.groupby(['timestamp', 'station_id']).first().reset_index()
df[(df['station_id'] == 'station_1') & (df['timestamp'] == date(2025, 1, 1))]

C:\Users\zamko\AppData\Local\Temp\ipykernel_14948\1243527087.py:3: PerformanceWarning: dropping on a non-lexsorted multi-index without a level parameter may impact performance.
  df.groupby('index').last()


,timestamp,station_id,num_bikes_available,num_ebikes_available
0,2025-01-01,station_1,12,4
10,2025-01-01,station_1,12,4
20,2025-01-01,station_1,14,5
30,2025-01-01,station_1,14,5
40,2025-01-01,station_1,13,5
50,2025-01-01,station_1,15,5
60,2025-01-01,station_1,14,4
70,2025-01-01,station_1,13,4
80,2025-01-01,station_1,13,4
90,2025-01-01,station_1,13,3


In [131]:
df = mock.df_trips[['started_at', 'start_station_id', 'rideable_type']]
df['started_at'] = df['started_at'].dt.date
df.groupby(['started_at', 'start_station_id', 'rideable_type']).size().reset_index()[:10]

,started_at,start_station_id,rideable_type,0
0,2025-01-01,station_1,classic_bike,11
1,2025-01-01,station_1,electric_bike,6
2,2025-01-01,station_10,classic_bike,16
3,2025-01-01,station_10,electric_bike,4
4,2025-01-01,station_2,classic_bike,15
5,2025-01-01,station_2,electric_bike,5
6,2025-01-01,station_3,classic_bike,10
7,2025-01-01,station_3,electric_bike,4
8,2025-01-01,station_4,classic_bike,11
9,2025-01-01,station_4,electric_bike,9


In [133]:
df = mock.df_trips[['started_at', 'end_station_id', 'rideable_type']]
df['started_at'] = df['started_at'].dt.date
df.groupby(['started_at', 'end_station_id', 'rideable_type']).size().reset_index()[:10]

,started_at,end_station_id,rideable_type,0
0,2025-01-01,station_1,classic_bike,10
1,2025-01-01,station_1,electric_bike,3
2,2025-01-01,station_10,classic_bike,10
3,2025-01-01,station_10,electric_bike,8
4,2025-01-01,station_2,classic_bike,15
5,2025-01-01,station_2,electric_bike,7
6,2025-01-01,station_3,classic_bike,21
7,2025-01-01,station_3,electric_bike,3
8,2025-01-01,station_4,classic_bike,8
9,2025-01-01,station_4,electric_bike,5


In [132]:
mock.df_trips

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
0,dab6996f-2b97-4bd9-8b94-d80b8a816d0c,classic_bike,2025-01-01 00:04:00,2025-01-01 00:11:00,E 69 St & Park Ave,station_8,W 65 St & 7 Ave,station_3,40.765627,-73.992517,40.738649,-73.938966,casual
1,268d5f58-077e-40d7-a9b0-a07d9962ede6,electric_bike,2025-01-01 00:15:00,2025-01-01 00:39:00,E 69 St & Park Ave,station_8,W 18 St & Broadway,station_10,40.765627,-73.992517,40.800567,-74.011832,casual
2,3d3d194c-fc79-47d8-9fde-da2c31608892,classic_bike,2025-01-01 00:20:00,2025-01-01 00:25:00,E 47 St & 3 Ave,station_4,W 75 St & Broadway,station_2,40.746683,-73.989392,40.854152,-73.921752,member
3,841fb9d7-aa64-4967-a401-dfef52578336,electric_bike,2025-01-01 00:21:00,2025-01-01 00:48:00,E 47 St & 3 Ave,station_4,W 18 St & Broadway,station_10,40.746683,-73.989392,40.800567,-74.011832,member
4,5887991b-9b93-4bb9-8193-7552addefcea,electric_bike,2025-01-01 00:29:00,2025-01-01 00:45:00,E 46 St & 1 Ave,station_5,E 69 St & Park Ave,station_8,40.764520,-73.921806,40.765627,-73.992517,member
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1160,7933cea1-75c6-4c2b-8fbe-cff6c4404e2f,classic_bike,2025-01-07 22:27:00,2025-01-07 22:54:00,E 47 St & 3 Ave,station_4,E 27 St & 7 Ave,station_9,40.746683,-73.989392,40.720844,-73.941276,member
1161,98bb9af5-763a-430b-a199-c0473f200ff9,classic_bike,2025-01-07 22:44:00,2025-01-07 23:04:00,W 75 St & Broadway,station_2,E 27 St & 7 Ave,station_9,40.854152,-73.921752,40.720844,-73.941276,member
1162,e4d1fd76-7493-486b-9bd3-8ee858eb132e,classic_bike,2025-01-07 22:46:00,2025-01-07 23:04:00,E 27 St & 7 Ave,station_9,W 18 St & Broadway,station_10,40.720844,-73.941276,40.800567,-74.011832,member
1163,81b25267-4239-4b8b-8b86-cbf385756076,electric_bike,2025-01-07 22:49:00,2025-01-07 23:02:00,W 18 St & Broadway,station_10,E 47 St & 3 Ave,station_4,40.800567,-74.011832,40.746683,-73.989392,casual


In [130]:
df = mock.df_inventory_ts[('station_1', 'electric_bike')].reset_index()
df['index'] = df['index'].dt.date
df.groupby('index').last()

C:\Users\zamko\AppData\Local\Temp\ipykernel_14948\2402040523.py:3: PerformanceWarning: dropping on a non-lexsorted multi-index without a level parameter may impact performance.
  df.groupby('index').last()


,station_1
,electric_bike
index,
2025-01-01,2
2025-01-02,6
2025-01-03,8
2025-01-04,8
2025-01-05,6
2025-01-06,5
2025-01-07,8


In [135]:
df = mock.df_inventory_ts[('station_1', 'electric_bike')].reset_index()
df['index'] = df['index'].dt.date
df.groupby('index').first()

C:\Users\zamko\AppData\Local\Temp\ipykernel_14948\634879567.py:3: PerformanceWarning: dropping on a non-lexsorted multi-index without a level parameter may impact performance.
  df.groupby('index').first()


,station_1
,electric_bike
index,
2025-01-01,4
2025-01-02,1
2025-01-03,7
2025-01-04,7
2025-01-05,8
2025-01-06,6
2025-01-07,5


In [ ]:
# 2025-01-01	4 first
# 2025-01-01	2 last
# 

In [127]:
mock.df_trips

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
0,dab6996f-2b97-4bd9-8b94-d80b8a816d0c,classic_bike,2025-01-01 00:04:00,2025-01-01 00:11:00,E 69 St & Park Ave,station_8,W 65 St & 7 Ave,station_3,40.765627,-73.992517,40.738649,-73.938966,casual
1,268d5f58-077e-40d7-a9b0-a07d9962ede6,electric_bike,2025-01-01 00:15:00,2025-01-01 00:39:00,E 69 St & Park Ave,station_8,W 18 St & Broadway,station_10,40.765627,-73.992517,40.800567,-74.011832,casual
2,3d3d194c-fc79-47d8-9fde-da2c31608892,classic_bike,2025-01-01 00:20:00,2025-01-01 00:25:00,E 47 St & 3 Ave,station_4,W 75 St & Broadway,station_2,40.746683,-73.989392,40.854152,-73.921752,member
3,841fb9d7-aa64-4967-a401-dfef52578336,electric_bike,2025-01-01 00:21:00,2025-01-01 00:48:00,E 47 St & 3 Ave,station_4,W 18 St & Broadway,station_10,40.746683,-73.989392,40.800567,-74.011832,member
4,5887991b-9b93-4bb9-8193-7552addefcea,electric_bike,2025-01-01 00:29:00,2025-01-01 00:45:00,E 46 St & 1 Ave,station_5,E 69 St & Park Ave,station_8,40.764520,-73.921806,40.765627,-73.992517,member
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1160,7933cea1-75c6-4c2b-8fbe-cff6c4404e2f,classic_bike,2025-01-07 22:27:00,2025-01-07 22:54:00,E 47 St & 3 Ave,station_4,E 27 St & 7 Ave,station_9,40.746683,-73.989392,40.720844,-73.941276,member
1161,98bb9af5-763a-430b-a199-c0473f200ff9,classic_bike,2025-01-07 22:44:00,2025-01-07 23:04:00,W 75 St & Broadway,station_2,E 27 St & 7 Ave,station_9,40.854152,-73.921752,40.720844,-73.941276,member
1162,e4d1fd76-7493-486b-9bd3-8ee858eb132e,classic_bike,2025-01-07 22:46:00,2025-01-07 23:04:00,E 27 St & 7 Ave,station_9,W 18 St & Broadway,station_10,40.720844,-73.941276,40.800567,-74.011832,member
1163,81b25267-4239-4b8b-8b86-cbf385756076,electric_bike,2025-01-07 22:49:00,2025-01-07 23:02:00,W 18 St & Broadway,station_10,E 47 St & 3 Ave,station_4,40.800567,-74.011832,40.746683,-73.989392,casual


In [119]:
df = mock.df_trips[['started_at', 'start_station_id', 'rideable_type']]
df['started_at'] = df['started_at'].dt.date
df.groupby(['started_at', 'start_station_id', 'rideable_type']).size().reset_index()[:10]

,started_at,start_station_id,rideable_type,0
0,2025-01-01,station_1,classic_bike,11
1,2025-01-01,station_1,electric_bike,6
2,2025-01-01,station_10,classic_bike,16
3,2025-01-01,station_10,electric_bike,4
4,2025-01-01,station_2,classic_bike,15
5,2025-01-01,station_2,electric_bike,5
6,2025-01-01,station_3,classic_bike,10
7,2025-01-01,station_3,electric_bike,4
8,2025-01-01,station_4,classic_bike,11
9,2025-01-01,station_4,electric_bike,9


In [88]:
known_ids = set(entities.station_ids) | set(entities.depot_ids)
result: dict[str, pd.DataFrame | None] = {}

# ── trips → observed_flow ────────────────────────────────────
df_trips = loader._source.df_trips
if df_trips is not None and not df_trips.empty:
    trips = df_trips[[
        "start_station_id", "end_station_id", "started_at", "rideable_type",
    ]].copy()
    trips = trips.rename(columns={
        "start_station_id": "source_id",
        "end_station_id": "target_id",
    })
    trips["date"] = pd.to_datetime(trips["started_at"]).dt.date
    trips["commodity_category"] = trips["rideable_type"]
    trips["quantity"] = 1.0
    trips["quantity_unit"] = "bike"
    trips["modal_type"] = None
    trips["resource_id"] = None

    mask = trips["source_id"].isin(known_ids) & trips["target_id"].isin(known_ids)
    trips = trips.loc[mask]

    if not trips.empty:
        grain = ["source_id", "target_id", "commodity_category", "date"]
        agg = trips.groupby(grain, as_index=False).agg(
            quantity=("quantity", "sum"),
            quantity_unit=("quantity_unit", "first"),
            modal_type=("modal_type", "first"),
            resource_id=("resource_id", "first"),
        )
        result["observed_flow"] = agg


# ── telemetry → observed_inventory (per commodity_category) ──
df_tel = loader._source.df_telemetry_ts
if df_tel is not None and not df_tel.empty:
    tel_base = df_tel[["station_id", "timestamp", "num_bikes_available",
                        "num_ebikes_available"]].copy()

    # Electric bikes
    tel_e = tel_base[["station_id", "timestamp", "num_ebikes_available"]].copy()
    tel_e = tel_e.rename(columns={
        "station_id": "facility_id",
        "num_ebikes_available": "quantity",
    })
    tel_e["commodity_category"] = "electric_bike"

    # Classic bikes = total - ebike
    tel_c = tel_base[["station_id", "timestamp"]].copy()
    tel_c["quantity"] = (
        tel_base["num_bikes_available"] - tel_base["num_ebikes_available"]
    )
    tel_c = tel_c.rename(columns={"station_id": "facility_id"})
    tel_c["commodity_category"] = "classic_bike"

    tel = pd.concat([tel_e, tel_c], ignore_index=True)
    tel["date"] = pd.to_datetime(tel["timestamp"]).dt.date
    tel["quantity_unit"] = "bike"
    tel["quantity"] = tel["quantity"].astype(float)
    tel = tel.loc[tel["facility_id"].isin(known_ids)]

    if not tel.empty:
        grain = ["facility_id", "commodity_category", "date"]
        tel = tel.sort_values("timestamp")
        agg = tel.groupby(grain, as_index=False).agg(
            quantity=("quantity", "last"),
            quantity_unit=("quantity_unit", "first"),
        )
        result["observed_inventory"] = agg

In [90]:
df = loader._source.df_telemetry_ts
df

,timestamp,region_id,lat,lon,station_id,capacity,short_name,name,num_docks_available,is_returning,last_reported,num_bikes_available,is_installed,is_renting,num_ebikes_available,num_docks_disabled,num_bikes_disabled
0,2025-01-01 00:00:00,5,40.814057,-73.973170,station_1,49,W 54,E 17 St & 2 Ave,37,1,2025-01-01 00:00:00,12,1,1,4,0,0
1,2025-01-01 00:00:00,1,40.854152,-73.921752,station_2,23,W 92,W 75 St & Broadway,21,1,2025-01-01 00:00:00,2,1,1,1,0,0
2,2025-01-01 00:00:00,5,40.738649,-73.938966,station_3,31,W 72,W 65 St & 7 Ave,17,1,2025-01-01 00:00:00,14,1,1,4,0,0
3,2025-01-01 00:00:00,1,40.746683,-73.989392,station_4,36,E 74,E 47 St & 3 Ave,12,1,2025-01-01 00:00:00,24,1,1,6,0,1
4,2025-01-01 00:00:00,4,40.764520,-73.921806,station_5,37,E 70,E 46 St & 1 Ave,12,1,2025-01-01 00:00:00,24,1,1,3,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1675,2025-01-07 23:00:00,4,40.714105,-73.925381,station_6,16,W 76,E 82 St & Park Ave,10,1,2025-01-07 23:00:00,6,1,1,1,0,0
1676,2025-01-07 23:00:00,4,40.703386,-73.979628,station_7,39,E 53,W 17 St & 5 Ave,4,1,2025-01-07 23:00:00,35,1,1,7,0,1
1677,2025-01-07 23:00:00,4,40.765627,-73.992517,station_8,21,E 20,E 69 St & Park Ave,11,1,2025-01-07 23:00:00,10,1,1,4,0,0
1678,2025-01-07 23:00:00,3,40.720844,-73.941276,station_9,48,W 81,E 27 St & 7 Ave,33,1,2025-01-07 23:00:00,15,1,1,4,0,0


In [89]:
df = result["observed_inventory"]
df = df[['facility_id', 'commodity_category', 'date', 'quantity']]
# Caclulate number of trips per day
df = (
    df.groupby(["date", "commodity_category", "facility_id"])
    .sum()
    .reset_index()
    .sort_values(["date", "commodity_category"])
)
cond_1 = (df['commodity_category'] == 'electric_bike')
cond_2 = (df['facility_id'] == 'station_1')
df[cond_1 & cond_2]

,date,commodity_category,facility_id,quantity
10,2025-01-01,electric_bike,station_1,2.0
30,2025-01-02,electric_bike,station_1,6.0
50,2025-01-03,electric_bike,station_1,8.0
70,2025-01-04,electric_bike,station_1,8.0
90,2025-01-05,electric_bike,station_1,6.0
110,2025-01-06,electric_bike,station_1,5.0
130,2025-01-07,electric_bike,station_1,8.0


In [82]:
df

,facility_id,commodity_category,date,quantity,quantity_unit
0,station_1,classic_bike,2025-01-01,7.0,bike
1,station_1,classic_bike,2025-01-02,0.0,bike
2,station_1,classic_bike,2025-01-03,1.0,bike
3,station_1,classic_bike,2025-01-04,1.0,bike
4,station_1,classic_bike,2025-01-05,3.0,bike
...,...,...,...,...,...
135,station_9,electric_bike,2025-01-03,1.0,bike
136,station_9,electric_bike,2025-01-04,0.0,bike
137,station_9,electric_bike,2025-01-05,0.0,bike
138,station_9,electric_bike,2025-01-06,0.0,bike


In [77]:
df = result['observed_flow']
df = df[['source_id', 'commodity_category', 'date', 'quantity']]
# Caclulate number of trips per day
df = (
    df.groupby(["date", "commodity_category", "source_id"])
    .sum()
    .reset_index()
    .sort_values(["date", "commodity_category"])
)
cond_1 = (df['commodity_category'] == 'electric_bike')
cond_2 = (df['source_id'] == 'station_1')
df[cond_1 & cond_2]

,date,commodity_category,source_id,quantity
10,2025-01-01,electric_bike,station_1,6.0
30,2025-01-02,electric_bike,station_1,4.0
50,2025-01-03,electric_bike,station_1,5.0
70,2025-01-04,electric_bike,station_1,7.0
90,2025-01-05,electric_bike,station_1,8.0
110,2025-01-06,electric_bike,station_1,6.0
130,2025-01-07,electric_bike,station_1,1.0


In [53]:
df = loader._source.df_trips.copy()
df = df[['start_station_id', 'end_station_id', 'rideable_type', 'started_at', 'ended_at']]
# Caclulate number of trips per day
df = (
    df.groupby([pd.Grouper(key="started_at", freq="D"), "rideable_type", "start_station_id"])
    .size()
    .reset_index(name="num_trips")
    .sort_values(["started_at", "rideable_type"])
)
cond_1 = (df['rideable_type'] == 'electric_bike')
cond_2 = (df['start_station_id'] == 'station_1')
df[cond_1 & cond_2]

,started_at,rideable_type,start_station_id,num_trips
10,2025-01-01,electric_bike,station_1,6
30,2025-01-02,electric_bike,station_1,4
50,2025-01-03,electric_bike,station_1,5
70,2025-01-04,electric_bike,station_1,7
90,2025-01-05,electric_bike,station_1,8
110,2025-01-06,electric_bike,station_1,6
130,2025-01-07,electric_bike,station_1,1


In [78]:
df = resolved.demand
cond_1 = df['facility_id'] == 'station_1'
cond_2 = df['commodity_category'] == 'electric_bike'
cond_3 = df['period_id'] == 'p0'
df[cond_1 & cond_2 & cond_3]

,facility_id,commodity_category,period_id,quantity
7,station_1,electric_bike,p0,6.0


In [24]:
from gbp.consumers.simulator import (
    ArrivalsPhase,
    DemandPhase,
    Environment,
    EnvironmentConfig,
)

config = EnvironmentConfig(
    phases=[DemandPhase(), ArrivalsPhase()],
    seed=42,
    scenario_id="workbook_run",
)

env = Environment(resolved, config)
while not env.is_done:
    if env.is_done:
        raise StopIteration("All periods have been processed.")

    period = env._periods[env._period_cursor]

    for phase in [DemandPhase(), ArrivalsPhase()]:
        print('here 1')
        if phase.should_run(period):
            result = phase.execute(env._state, env._resolved, period)
            a = 1/0
            env._state = result.state
            env._log.record_events(result, phase.name, period)

    env._log.record_period(env._state, period)

    # Advance to next period
    env._period_cursor += 1
    if not env.is_done:
        print('here 2')
        next_period = env._periods[env._period_cursor]
        env._state = env._state.advance_period(
            next_period_index=next_period.period_index,
            next_period_id=next_period.period_id,
        )

log_tables = simulation_log.to_dataframes()
inventory_log = log_tables["simulation_inventory_log"]
flow_log = log_tables["simulation_flow_log"]
unmet_demand_log = log_tables["simulation_unmet_demand_log"]

here 1


ZeroDivisionError: division by zero

In [37]:
df = resolved.observed_inventory
cond_1 = df['facility_id'] == 'station_1'
cond_2 = df['commodity_category'] == 'electric_bike'
cond_3 = df['period_id'] == 'p0'
df[cond_1 & cond_2 & cond_3]

,facility_id,commodity_category,period_id,quantity
7,station_1,electric_bike,p0,2.0


In [38]:
df = resolved.demand
cond_1 = df['facility_id'] == 'station_1'
cond_2 = df['commodity_category'] == 'electric_bike'
cond_3 = df['period_id'] == 'p0'
df[cond_1 & cond_2 & cond_3]

,facility_id,commodity_category,period_id,quantity
7,station_1,electric_bike,p0,6.0


In [19]:
df = inventory_log
cond_1 = df['facility_id'] == 'station_1'
cond_2 = df['commodity_category'] == 'electric_bike'
cond_3 = df['period_id'] == 'p0'
df[cond_1 & cond_2 & cond_3]

,period_index,period_id,facility_id,commodity_category,quantity
0,0,p0,station_1,electric_bike,0.0


In [8]:
# %% Gradio Interactive Dashboard
# Sliders for mock config + period selector → map with facilities & inventory over time
# Generate mock → browse observed inventory by period
# Run simulation → compare historical vs simulated inventory at each period

import gradio as gr
import plotly.graph_objects as go
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def _build_pipeline(config: dict):
    """Generate mock → build RawModelData → ResolvedModelData."""
    from gbp.loaders import DataLoaderMock, DataLoaderGraph, GraphLoaderConfig

    mock = DataLoaderMock(config)
    # NOTE: do NOT call mock.load_data() here — DataLoaderGraph.load_data()
    # calls self._source.load_data() internally.  Calling it twice advances
    # the RNG past seed and produces different (wrong) data.
    loader = DataLoaderGraph(mock, GraphLoaderConfig())
    loader.load_data()
    return loader.resolved


def _run_simulation(resolved) -> pd.DataFrame:
    """Run Environment and return simulation_inventory_log."""
    from gbp.consumers.simulator import (
        ArrivalsPhase, DemandPhase, Environment, EnvironmentConfig,
    )
    env_config = EnvironmentConfig(
        phases=[DemandPhase(), ArrivalsPhase()],
        seed=42,
        scenario_id="gradio_run",
    )
    env = Environment(resolved, env_config)
    log = env.run()
    tables = log.to_dataframes()
    return tables["simulation_inventory_log"]


def _inventory_at_period(df: pd.DataFrame, period_id: str) -> pd.DataFrame:
    """Sum quantity per facility at a given period_id."""
    if df is None or df.empty:
        return pd.DataFrame(columns=["facility_id", "quantity"])
    snapshot = df[df["period_id"] == period_id]
    if snapshot.empty:
        return pd.DataFrame(columns=["facility_id", "quantity"])
    return snapshot.groupby("facility_id", as_index=False)["quantity"].sum()


def _make_map(
    facilities: pd.DataFrame,
    observed_inventory: pd.DataFrame,
    title: str,
    simulated_inventory: pd.DataFrame | None = None,
) -> go.Figure:
    """Single-marker map: one dot per facility, hover shows observed + optional simulated."""
    merged = facilities.drop_duplicates("facility_id").merge(
        observed_inventory.rename(columns={"quantity": "observed"}),
        on="facility_id",
        how="left",
    )
    merged["observed"] = merged["observed"].fillna(0)

    if simulated_inventory is not None and not simulated_inventory.empty:
        merged = merged.merge(
            simulated_inventory.rename(columns={"quantity": "simulated"}),
            on="facility_id",
            how="left",
        )
        merged["simulated"] = merged["simulated"].fillna(0)
        merged["delta"] = merged["simulated"] - merged["observed"]
        has_sim = True
    else:
        merged["simulated"] = np.nan
        merged["delta"] = np.nan
        has_sim = False

    color_map = {"station": "#1f77b4", "depot": "#d62728"}
    merged["marker_size"] = np.clip(merged["observed"] * 0.5 + 6, 6, 40)

    if has_sim:
        hover_tpl = (
            "<b>%{customdata[0]}</b><br>"
            "observed: %{customdata[1]:.0f}<br>"
            "simulated: %{customdata[2]:.0f}<br>"
            "delta: %{customdata[3]:+.0f}"
            "<extra></extra>"
        )
    else:
        hover_tpl = (
            "<b>%{customdata[0]}</b><br>"
            "inventory: %{customdata[1]:.0f}"
            "<extra></extra>"
        )

    fig = go.Figure()
    for facility_type, group in merged.groupby("facility_type"):
        customdata = np.stack(
            [
                group["name"],
                group["observed"],
                group["simulated"] if has_sim else group["observed"],
                group["delta"] if has_sim else np.zeros(len(group)),
            ],
            axis=-1,
        )

        text_col = group["observed"].round(0).astype(int).astype(str)
        if has_sim:
            text_col = (
                text_col + " / "
                + group["simulated"].round(0).astype(int).astype(str)
            )

        fig.add_trace(go.Scattermap(
            lat=group["lat"],
            lon=group["lon"],
            mode="markers+text",
            marker=dict(
                size=group["marker_size"],
                color=color_map.get(facility_type, "#7f7f7f"),
            ),
            text=text_col,
            textposition="top center",
            customdata=customdata,
            hovertemplate=hover_tpl,
            name=str(facility_type),
        ))

    center_lat = merged["lat"].mean()
    center_lon = merged["lon"].mean()
    fig.update_layout(
        title=title,
        map=dict(
            style="open-street-map",
            center=dict(lat=center_lat, lon=center_lon),
            zoom=11,
        ),
        height=600,
        margin=dict(l=0, r=0, t=40, b=0),
        showlegend=True,
    )
    return fig


# ---------------------------------------------------------------------------
# Gradio state
# ---------------------------------------------------------------------------
_state = {
    "resolved": None,
    "facilities": None,
    "observed_inv": None,
    "sim_log": None,
    "period_map": {},
}


def generate_mock(n_stations, n_depots, n_timestamps, ebike_fraction, seed):
    config = {
        "n_stations": int(n_stations),
        "n_depots": int(n_depots),
        "n_timestamps": int(n_timestamps),
        "time_freq": "h",
        "start_date": "2025-01-01",
        "num_resources": 3,
        "resource_capacity": 100,
        "ebike_fraction": float(ebike_fraction),
        "depot_capacity": 200,
        "seed": int(seed),
    }
    resolved = _build_pipeline(config)
    facilities = resolved.facilities[["facility_id", "facility_type", "name", "lat", "lon"]].copy()
    observed_inv = resolved.observed_inventory

    period_df = resolved.periods[["period_id", "start_date"]].sort_values("period_id")
    period_map = dict(zip(period_df["period_id"], period_df["start_date"].astype(str)))

    _state["resolved"] = resolved
    _state["facilities"] = facilities
    _state["observed_inv"] = observed_inv
    _state["sim_log"] = None
    _state["period_map"] = period_map

    n_periods = len(period_map)
    first_pid = list(period_map.keys())[0]
    first_label = f"{first_pid} ({period_map[first_pid]})"

    observed_at_period = _inventory_at_period(observed_inv, first_pid)
    fig = _make_map(facilities, observed_at_period, f"Observed Inventory — {first_label}")
    status = f"Generated: {len(facilities)} facilities, {n_periods} periods"

    slider_update = gr.update(minimum=0, maximum=n_periods - 1, value=0, visible=True)
    return fig, status, slider_update


def on_period_change(period_idx):
    period_idx = int(period_idx)
    period_ids = list(_state["period_map"].keys())
    if not period_ids or _state["facilities"] is None:
        return go.Figure(), "No data"

    period_idx = min(period_idx, len(period_ids) - 1)
    pid = period_ids[period_idx]
    label = f"{pid} ({_state['period_map'][pid]})"

    observed_at_period = _inventory_at_period(_state["observed_inv"], pid)
    simulated_at_period = (
        _inventory_at_period(_state["sim_log"], pid)
        if _state["sim_log"] is not None
        else None
    )

    title = (
        f"Historical vs Simulated — {label}"
        if simulated_at_period is not None
        else f"Observed Inventory — {label}"
    )
    fig = _make_map(_state["facilities"], observed_at_period, title, simulated_at_period)
    return fig, f"Period {period_idx}: {label}"


def run_simulation():
    if _state["resolved"] is None:
        return go.Figure(), "No data — click 'Generate Mock Data' first"

    sim_log = _run_simulation(_state["resolved"])
    _state["sim_log"] = sim_log

    period_ids = list(_state["period_map"].keys())
    first_pid = period_ids[0]
    label = f"{first_pid} ({_state['period_map'][first_pid]})"

    observed_at_period = _inventory_at_period(_state["observed_inv"], first_pid)
    simulated_at_period = _inventory_at_period(sim_log, first_pid)
    fig = _make_map(
        _state["facilities"],
        observed_at_period,
        f"Historical vs Simulated — {label}",
        simulated_at_period,
    )
    status = f"Simulation complete: {len(sim_log)} log rows. Use period slider to browse."
    return fig, status


# ---------------------------------------------------------------------------
# Gradio UI
# ---------------------------------------------------------------------------
with gr.Blocks(title="Bike-Share Simulation Dashboard") as demo:
    gr.Markdown("## Bike-Share Network — Mock Generation & Simulation")

    with gr.Row():
        with gr.Column(scale=1):
            sl_stations = gr.Slider(3, 50, value=10, step=1, label="Stations")
            sl_depots = gr.Slider(1, 5, value=2, step=1, label="Depots")
            sl_timestamps = gr.Slider(24, 720, value=168, step=24, label="Timestamps (hours)")
            sl_ebike = gr.Slider(0.0, 1.0, value=0.3, step=0.05, label="E-bike fraction")
            sl_seed = gr.Slider(0, 999, value=42, step=1, label="Seed")
            btn_generate = gr.Button("Generate Mock Data", variant="primary")
            btn_simulate = gr.Button("Run Simulation", variant="secondary")
            sl_period = gr.Slider(0, 1, value=0, step=1, label="Period", visible=False)
            status_text = gr.Textbox(label="Status", interactive=False)

        with gr.Column(scale=3):
            map_plot = gr.Plot(label="Network Map")

    btn_generate.click(
        fn=generate_mock,
        inputs=[sl_stations, sl_depots, sl_timestamps, sl_ebike, sl_seed],
        outputs=[map_plot, status_text, sl_period],
    )
    btn_simulate.click(
        fn=run_simulation,
        inputs=[],
        outputs=[map_plot, status_text],
    )
    sl_period.change(
        fn=on_period_change,
        inputs=[sl_period],
        outputs=[map_plot, status_text],
    )

demo.launch()

e:\Documents\vlzm\GFDRR\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [9]:
# Моя идея была в том, что в inventory_observed и в inventory_simulator должны по идее быть одни и те же значения в тривиальном случае, когда симулятор просто отрабатывает историю без каких либо изменений.
# Более того, сейчас в UI после исполнения "generate mock data", когда я навожу на node - я вижу station_i или depot_i два раза.
# А после того как отрабатывает "run simulation" у каждой станции и каждого депота появляется по две node. Одна историческая - другая симулированая. Так вот, это же должна быть одна node. 
# И в ней должны быть "facility name", "inventory historical", "inventory simulator". 

2026-04-06 21:30:23 [info     ] load_start                     loader=graph_core
2026-04-06 21:30:25 [debug    ] source_validated               loader=graph_core
2026-04-06 21:30:25 [debug    ] observed_flow_built            loader=graph_core rows=723
2026-04-06 21:30:25 [debug    ] observed_inventory_built       loader=graph_core rows=140
2026-04-06 21:30:25 [info     ] load_done                      edges=132 facilities=12 loader=graph_core
